In [0]:
# Date-range presets for data loading
DATE_RANGES = {
    # "full": {"start": "2022-06-30", "end": "2024-07-01", "mob_months": 25},
     "full": {"start": "2022-06-30", "end": "2025-07-01", "mob_months": 13},
    "recent": {"start": "2025-01-31", "end": "2026-07-01", "mob_months": 37},
}

# Bad-definition candidates (min_bucket follows the DELINQUENCYBUCKET encoding:
#   0=current, 1=1-29 DPD, 2=30-59 DPD, 3=60-89 DPD, 4=90+ DPD)
BAD_DEFINITIONS = {
    "Ever60DPD_12M": {"min_bucket": 3, "window_months": 12},
    "Ever30DPD_6M":  {"min_bucket": 2, "window_months": 6},
    "Ever90DPD_12M": {"min_bucket": 4, "window_months": 12},
}

# Target generation: 12-month performance window
TARGET_WINDOW_MONTHS = 12

# Minimum OPEN snapshots a card must have to be retained
# MIN_OPEN_SNAPSHOTS = 3

# Columns needed downstream (avoids OOM when converting full 88-col frame to pandas)
ANALYSIS_COLS = [
    "SNAPSHOT_DATE", "CRNNO", "CIF", "CREDITCARDNO", "APPROVEDATE", "APPRECDDATE",
    "DELINQUENCYBUCKET", "CURRENT_STATUS", "REFERENCE_NUMBER", "FTS_Ref_No",
    "EMID_REFERENCENUMBER", "GENERALSTAT",
    # Filter / exclusion columns (kept for downstream analysis)
    "CARD_BUSINESS_SEGMENT", "CUSTOMER_SEGMENT", "VIP", "RULING_FAMILY_FLAG",
    "PRIMARYCARD", "F_SEGMENT_NEW", "SEGMENT_POLICY", "CLASSCODE", "ACTIVE_REPORTING",
]

ID_COLS = [
    "CRNNO", "CIF", "CREDITCARDNO", "APPROVEDATE", "APPRECDDATE",
    "REFERENCE_NUMBER", "FTS_Ref_No", "EMID_REFERENCENUMBER", "GENERALSTAT",
    # Filter / exclusion columns (carried through the pipeline)
    "CARD_BUSINESS_SEGMENT", "CUSTOMER_SEGMENT", "VIP", "RULING_FAMILY_FLAG",
    "PRIMARYCARD", "F_SEGMENT_NEW", "SEGMENT_POLICY", "CLASSCODE", "ACTIVE_REPORTING",
]

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col

In [0]:
def load_and_clean_credit_card_data(date_start, date_end, mob_months=37):
    """
    Load credit card history and apply business exclusion filters.

    Parameters
    ----------
    date_start : str   – APPROVEDATE lower bound (exclusive), e.g. '2022-06-30'
    date_end   : str   – APPROVEDATE upper bound (exclusive), e.g. '2025-07-01'
    mob_months : int   – months-on-book observation window (default 37)

    Returns
    -------
    pyspark.sql.DataFrame  – cleaned Spark DataFrame with exclusions applied.
    """
    # Step 1: Load raw data within the observation window
    df = spark.sql(f"""
        SELECT *
        FROM mis_automation.silver_credit_card.vw_credit_card_history
        WHERE SNAPSHOT_DATE >= APPROVEDATE
          AND SNAPSHOT_DATE < add_months(APPROVEDATE, {mob_months})
          AND APPROVEDATE > '{date_start}'
          AND APPROVEDATE < '{date_end}'
          AND UPPER(TRIM(CARD_BUSINESS_SEGMENT)) IN ('CARDS')
          AND UPPER(TRIM(CUSTOMER_SEGMENT)) = 'SALARIED'
    """)

    # Step 2: Exclusion conditions
    exclude_cond = (
        (F.col("VIP") == "1")                                                             # 3a: VIP customers
        | (F.upper(F.trim(F.col("RULING_FAMILY_FLAG"))) == "Y")                             # 3b: Ruling family
        | (F.col("PRIMARYCARD") == 0)                                                      # 7: Supplementary cards
        | (F.upper(F.trim(F.col("F_SEGMENT_NEW"))) == "SALARIED-IM")                        # 10a: Salaried-IM
        | (F.upper(F.trim(F.col("SEGMENT_POLICY"))).isin(
              "PL CUSTOMERS", "AUTO-LOAN CUSTOMERS", "MORTGAGE CXMS", "STAFF"))          # 10b: Unwanted segment policies
        |(F.upper(F.trim(F.col("GENERALSTAT"))).isin('FRUD')))
     
    inclusive_cond = (
        (F.col("CLASSCODE").isNull()| ~F.trim(F.col("CLASSCODE")).isin("STAFF", "SME"))   # 1, 8: SME/STAFF class code
        & (F.upper(F.trim(F.col("CARD_BUSINESS_SEGMENT"))).isin("CARDS"))                  # 2: CARDS segment
        & (F.upper(F.trim(F.col("CUSTOMER_SEGMENT"))) == "SALARIED")                       # 6: Salaried
        & (F.upper(F.trim(F.col("ACTIVE_REPORTING"))) == "Y")                              # 4: Active reporting
    )        

    # df_clean = df.filter(~exclude_cond).filter(inclusive_cond)  # disabled
    df_clean = df  # Exclusion/inclusion filters skipped

    return df_clean

def generate_target(df, window_months=TARGET_WINDOW_MONTHS):
    """
    Assign one target label per CREDITCARDNO based on the max
    DELINQUENCYBUCKET within the first `window_months` after approval.

    Target mapping
    --------------
    -2 : immature (fewer than `window_months` unique snapshot months)
     0 : good    (max bucket == 0)
    -1 : indeterminate (max bucket in 1-3)
     1 : bad     (max bucket >= 4, i.e. 90+ DPD)
    """
    df = df.copy()
    df["SNAPSHOT_DATE"] = pd.to_datetime(df["SNAPSHOT_DATE"])
    df["APPROVEDATE"]   = pd.to_datetime(df["APPROVEDATE"])

    mask = (
        (df["SNAPSHOT_DATE"] >= df["APPROVEDATE"])
        & (df["SNAPSHOT_DATE"] < df["APPROVEDATE"] + pd.DateOffset(months=window_months))
    )
    window_df = df.loc[mask].copy()
    window_df["_snap_month"] = window_df["SNAPSHOT_DATE"].dt.to_period("M")

    agg = (
        window_df
        .groupby("CREDITCARDNO")
        .agg(max_bucket=("DELINQUENCYBUCKET", "max"),
             unique_months=("_snap_month", "nunique"))
        .reset_index()
    )

    conditions = [
        agg["unique_months"] < window_months,  # -2: immature
        agg["max_bucket"] >= 4,                 #  1: bad 3 is used
        agg["max_bucket"] == 0,                 #  0: good
        agg["max_bucket"].between(1, 3),        # -1: indeterminate
    ]
    agg["target"] = np.select(conditions, [-2, 1, 0, -1], default=np.nan)

    if agg["target"].isna().any():
        n_drop = agg["target"].isna().sum()
        print(f"WARNING: {n_drop} rows matched no target condition — dropped.")
        agg = agg.dropna(subset=["target"])
    agg["target"] = agg["target"].astype(int)

    id_df = df[ID_COLS].drop_duplicates(subset="CREDITCARDNO")
    result = id_df.merge(agg[["CREDITCARDNO", "target"]], on="CREDITCARDNO", how="inner")
    result["REFERENCE_NUMBER"] = (
        result["REFERENCE_NUMBER"].astype(str).str.replace(r'\.0$', '', regex=True)
    )
    return result

def load_aecb_feature_store(df_target):
    """
    Load the AECB feature-store table, filter to reference numbers
    present in `df_target`, merge, and split into thin-file subset.

    Optimised: uses a broadcast Spark join instead of .isin() with a
    Python list, which pushes the filter into the scan and avoids
    serialising thousands of reference numbers to every executor.

    Returns
    -------
    aecb_merged : pd.DataFrame – AECB features joined with target
    df_thin     : pd.DataFrame – subset where file_type == 'Thin File'
    """
    # ── 1. Build a small Spark DF of unique reference numbers ────────────
    ref_pd = (
        df_target[["REFERENCE_NUMBER"]]
        .drop_duplicates()
        .rename(columns={"REFERENCE_NUMBER": "ref_num"})
    )
    ref_pd["ref_num"] = ref_pd["ref_num"].astype(str)
    ref_spark = spark.createDataFrame(ref_pd)
    ref_spark.createOrReplaceTempView("_aecb_ref_nums")

    # ── 2. Push filter into Spark via broadcast join ────────────────────
    aecb_filtered = spark.sql("""
        SELECT /*+ BROADCAST(r) */ a.*
        FROM mis_automation.silver_bureauone.slv_credit_card_aecb_feature_store a
        INNER JOIN _aecb_ref_nums r
               ON CAST(a.Reference_Number AS STRING) = r.ref_num
    """)
    aecb_pd = aecb_filtered.toPandas()

    # ── 2b. Prefix AECB feature columns with 'aecb_' ───────────────────
    skip_cols = {"Reference_Number"}  # columns used for joining — don't rename
    aecb_pd = aecb_pd.rename(
        columns={c: f"aecb_{c}" for c in aecb_pd.columns if c not in skip_cols}
    )

    # ── 3. Pandas merge with target columns ─────────────────────────────
    aecb_pd["Reference_Number"] = aecb_pd["Reference_Number"].astype(str)
    df_target["REFERENCE_NUMBER"] = df_target["REFERENCE_NUMBER"].astype(str)

    aecb_merged = aecb_pd.merge(
        df_target[ID_COLS + ["target"]].drop_duplicates(),
        left_on="Reference_Number",
        right_on="REFERENCE_NUMBER",
        how="inner",
    )
    # ── 3. Use the new file type logic from policy ────────────────────
    df_thin = aecb_merged.loc[aecb_merged["aecb_file_type_revised"] == "Thin File"].reset_index(drop=True)

    print(f"AECB shape   : {aecb_pd.shape}")
    print(f"Merged shape : {aecb_merged.shape}")
    print(f"Thin-file    : {df_thin.shape[0]:,} rows  |  {df_thin['CREDITCARDNO'].nunique():,} cards")
    print(f"\nTarget distribution:\n{aecb_merged['target'].value_counts(dropna=False).sort_index()}")
    return aecb_merged, df_thin

def load_fts_features(df_thin_score, journey="digital"):
    """
    Load FTS features and merge them onto the thin-file scored DataFrame.

    Parameters
    ----------
    df_thin_score : pd.DataFrame
        Thin-file DataFrame with an 'FTS_Ref_No' column to join on.
    journey : str
        Which journey to load. Currently supports 'digital'.
        ('physical' to be added later.)

    Returns
    -------
    pd.DataFrame – df_thin_score enriched with FTS feature columns.
    """
    # ── 1. Select source table based on journey ──────────────────────────
    if journey == "digital":
        table = "mis_automation.silver_ftsconnect.fts_features"
    # elif journey == "physical":
    #     table = "<physical_journey_table>"   # TODO: add physical journey table
    else:
        raise ValueError(f"Unknown journey '{journey}'. Supported: 'digital'.")

    fts_features_sdf = spark.sql(f"SELECT * FROM {table}")

    print(f"Journey            : {journey}")
    print(f"df_thin_score rows : {len(df_thin_score):,}")
    print(f"fts_features rows  : {fts_features_sdf.count():,}")
    print(f"fts_features cols  : {len(fts_features_sdf.columns)}")

    # ── 2. Convert to Spark and left-join on FTS_Ref_No ──────────────────
    df_thin_score_sdf = spark.createDataFrame(df_thin_score)

    joined_sdf = df_thin_score_sdf.join(
        fts_features_sdf,
        df_thin_score_sdf["FTS_Ref_No"] == fts_features_sdf["fts_ref_no"],
        how="left",
    )
    joined_sdf = joined_sdf.drop(fts_features_sdf["fts_ref_no"])

    # ── 3. Collect back to pandas and report ─────────────────────────────
    existing_cols = set(df_thin_score.columns)
    df_merged = joined_sdf.toPandas()
    new_cols = [c for c in df_merged.columns if c not in existing_cols]

    print(f"\nMerged shape : {df_merged.shape}")
    if new_cols:
        print(f"Match rate   : {df_merged[new_cols[0]].notna().mean():.2%}")
        print(f"\nNew columns from fts_features ({len(new_cols)}):")
        for i, c in enumerate(new_cols, 1):
            print(f"  {i:3d}. {c}")
    else:
        print("No new columns added.")

    return df_merged

def load_fts_features_physical(df_thin_score):
    """
    EMID fallback: for rows without FTS_Ref_No, match FTS features
    via EMID_REFERENCENUMBER = fts_master_id_number.

    Keeps the earliest fts_statement_create_date per CREDITCARDNO
    within a [-7d, +30d) window around APPRECDDATE.

    Parameters
    ----------
    df_thin_score : pd.DataFrame
        Thin-file DataFrame (already enriched by the digital journey).
        Must contain 'FTS_Ref_No', 'EMID_REFERENCENUMBER', 'APPRECDDATE',
        and 'CREDITCARDNO'.

    Returns
    -------
    pd.DataFrame – df_thin_score with FTS columns back-filled for rows
                   that had no FTS_Ref_No but matched on EMID.
    """
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    # ── 0. Guard: ensure CREDITCARDNO is a column ────────────────────────
    if "CREDITCARDNO" not in df_thin_score.columns and df_thin_score.index.name == "CREDITCARDNO":
        df_thin_score = df_thin_score.reset_index()

    # Clean up leftover __emid columns from a previous partial run
    emid_leftover = [c for c in df_thin_score.columns if c.endswith("__emid")]
    if emid_leftover:
        df_thin_score.drop(columns=emid_leftover, inplace=True)
        print(f"Cleaned up {len(emid_leftover)} leftover __emid columns")

    # ── 1. Identify rows where FTS_Ref_No is not available ───────────────
    no_fts_mask = (
        df_thin_score["FTS_Ref_No"].isna()
        | df_thin_score["FTS_Ref_No"].astype(str).isin(["None", "", "nan"])
    )
    df_no_fts = (
        df_thin_score
        .loc[no_fts_mask, ["CREDITCARDNO", "EMID_REFERENCENUMBER", "APPRECDDATE","APPROVEDATE"]]
        .drop_duplicates(subset="CREDITCARDNO")
    )
    df_no_fts = df_no_fts[df_no_fts["EMID_REFERENCENUMBER"].notna()].copy()
    df_no_fts["EMID_REFERENCENUMBER"] = df_no_fts["EMID_REFERENCENUMBER"].astype(str)
    print(f"Cases without FTS_Ref_No (with valid EMID): {len(df_no_fts):,}")

    if df_no_fts.empty:
        print("Nothing to backfill — returning df_thin_score unchanged.")
        return df_thin_score

    # ── 1b. Overall EMID match-rate diagnostic ───────────────────────────
    all_emids = (
        df_thin_score
        .loc[df_thin_score["EMID_REFERENCENUMBER"].notna(),
             ["CREDITCARDNO", "EMID_REFERENCENUMBER"]]
        .drop_duplicates(subset="CREDITCARDNO")
    )
    all_emids["EMID_REFERENCENUMBER"] = all_emids["EMID_REFERENCENUMBER"].astype(str)
    all_emids_sdf = spark.createDataFrame(all_emids)

    fts_sdf_check = spark.sql(
        "SELECT DISTINCT fts_master_id_number FROM mis_automation.silver_ftsconnect.fts_features"
    )
    matched_emids = all_emids_sdf.join(
        fts_sdf_check,
        all_emids_sdf["EMID_REFERENCENUMBER"] == fts_sdf_check["fts_master_id_number"],
        how="inner",
    )
    matched_count = matched_emids.count()
    total_emid_count = all_emids_sdf.count()
    print(f"\n── EMID match check (all rows with valid EMID) ──")
    print(f"Total unique CREDITCARDNOs with valid EMID: {total_emid_count:,}")
    print(f"Found in fts_features.fts_master_id_number: {matched_count:,}")
    if total_emid_count:
        print(f"Match rate:                                 {matched_count / total_emid_count:.2%}")
    print(f"Not found:                                  {total_emid_count - matched_count:,}\n")

    # ── 2. Spark lookup DF ───────────────────────────────────────────────
    lookup_sdf = spark.createDataFrame(df_no_fts)

    # ── 3. Join fts_features on EMID ────────────────────────────────────
    fts_sdf = spark.sql("SELECT * FROM mis_automation.silver_ftsconnect.fts_features")
    joined = lookup_sdf.join(
        fts_sdf,
        lookup_sdf["EMID_REFERENCENUMBER"] == fts_sdf["fts_master_id_number"],
        how="inner",
    )

    cards_before_date_filter = joined.select("CREDITCARDNO").distinct().count()
    print(f"── Drop-off Diagnosis ──")
    print(f"Step 3 - Cards matched on EMID (before date filter): {cards_before_date_filter:,}")
    print(f"         Cards NOT found in fts_features at all:     {len(df_no_fts) - cards_before_date_filter:,}")

    # ── 4. Date filter: [-7d, +30d) around APPRECDDATE ──────────────────
    joined_filtered = joined.filter(
        (F.col("fts_statement_create_date") >= F.expr("APPRECDDATE - INTERVAL 60 DAYS"))
        & (F.col("fts_statement_create_date") < F.expr("APPRECDDATE + INTERVAL 30 DAYS"))
        & (F.col("fts_statement_create_date") < F.col("APPROVEDATE"))
    )
    cards_after_date_filter = joined_filtered.select("CREDITCARDNO").distinct().count()
    print(f"Step 4 - Cards surviving [-7d, +30d) date filter:    {cards_after_date_filter:,}")
    print(f"         Cards LOST by date filter:                  {cards_before_date_filter - cards_after_date_filter:,}\n")

    # ── 5. Keep latest fts_statement_create_date per card ───────────────
    w = Window.partitionBy("CREDITCARDNO").orderBy(F.col("fts_statement_create_date").desc())
    fts_emid = (
        joined_filtered
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn", "EMID_REFERENCENUMBER", "APPRECDDATE")
    )
    fts_emid_pd = fts_emid.toPandas()
    print(f"Matched via EMID: {fts_emid_pd['CREDITCARDNO'].nunique():,} cards")

    # ── 6. Back-fill FTS columns into df_thin_score ─────────────────────
    fts_cols = [
        c for c in fts_emid_pd.columns
        if c.startswith("fts_")
        and c != "fts_master_id_number"
        and c in df_thin_score.columns
    ]
    print(f"FTS feature columns to backfill: {len(fts_cols)}")

    fts_lookup = (
        fts_emid_pd.drop_duplicates(subset="CREDITCARDNO")
        .set_index("CREDITCARDNO")[fts_cols]
    )
    cards_to_fill = set(fts_lookup.index)
    fill_mask = no_fts_mask & df_thin_score["CREDITCARDNO"].isin(cards_to_fill)
    print(f"Rows to backfill: {fill_mask.sum():,}")

    for col_name in fts_cols:
        mapping = fts_lookup[col_name].to_dict()
        df_thin_score.loc[fill_mask, col_name] = (
            df_thin_score.loc[fill_mask, "CREDITCARDNO"].map(mapping)
        )

    # ── Summary ──────────────────────────────────────────────────────────
    first_fts = fts_cols[0] if fts_cols else None
    if first_fts:
        after = df_thin_score[first_fts].notna().sum()
        before = after - fill_mask.sum()
        n = len(df_thin_score)
        print(f"\nFTS coverage before EMID backfill: {before:,} / {n:,} ({before / n:.1%})")
        print(f"FTS coverage after  EMID backfill: {after:,} / {n:,} ({after / n:.1%})")
        print(f"Rows gained: {fill_mask.sum():,}")

    return df_thin_score

In [0]:
# ── Run the full pipeline ──────────────────────────────────────────────
cfg = DATE_RANGES["full"]

# 1. Load credit card data (exclusion/inclusion filters disabled)
df_spark = load_and_clean_credit_card_data(cfg["start"], cfg["end"], cfg["mob_months"])
print(f"Raw rows: {df_spark.count():,}")

# 2. Generate target labels
df_pd = df_spark.select(*ANALYSIS_COLS).toPandas()
df_target = generate_target(df_pd)
print(f"\nTarget shape: {df_target.shape}")
print(df_target["target"].value_counts().sort_index())

# 3. Load AECB feature store
aecb_merged, df_thin = load_aecb_feature_store(df_target)

# 4. FTS features – digital journey
df_thin = load_fts_features(df_thin, journey="digital")

# 5. FTS features – physical (EMID backfill)
df_thin = load_fts_features_physical(df_thin)

# 6. Drop duplicate columns (case-insensitive clash) and save
dupes = df_thin.columns[df_thin.columns.str.lower().duplicated(keep="first")]
if len(dupes):
    print(f"Dropping duplicate columns: {list(dupes)}")
    df_thin = df_thin.drop(columns=dupes)

print(f"\nFinal shape: {df_thin.shape}")
sdf = spark.createDataFrame(df_thin)

# 7. Drop NullType (void) columns — all-null cols that Parquet can't store/read
from pyspark.sql.types import NullType
void_cols = [f.name for f in sdf.schema.fields if isinstance(f.dataType, NullType)]
if void_cols:
    print(f"Dropping NullType (void) columns: {void_cols}")
    sdf = sdf.drop(*void_cols)

sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("mis_automation.ds_exploration.acard_cc_data")
print("Saved to mis_automation.ds_exploration.acard_cc_data")

In [0]:
df = spark.sql("SELECT * FROM mis_automation.ds_exploration.acard_cc_data LIMIT 5").toPandas()
df

In [0]:
[x for x in df.columns.tolist() if 'stdde' in x]

In [0]:
[c for c in df.columns.tolist() if 'variance' in c.lower()]

In [0]:
# Columns that are neither fts_ nor aecb_ prefixed
filtered_cols = [x for x in df.columns.tolist() if "fts_" not in x and "aecb_" not in x]

# Explicit ID / status columns to keep
extra_cols = [
    "CRNNO", "CIF", "CREDITCARDNO", "APPROVEDATE", "APPRECDDATE",
    "DELINQUENCYBUCKET", "CURRENT_STATUS", "REFERENCE_NUMBER",
    "FTS_Ref_No", "EMID_REFERENCENUMBER", "GENERALSTAT",
    "aecb_file_type_revised", "aecb_file_type", "aecb_file_type_new",
]

# Combine, deduplicate, and sort
final_cols = sorted(set(filtered_cols + extra_cols))
final_cols

In [0]:
spark.sql("SELECT Customer_number FROM mis_automation.silver_fin_db.customer_information LIMIT 5").display()

In [0]:
demo = spark.sql("SELECT * FROM mis_automation.silver_fin_db.customer_information LIMIT 5").toPandas()
alist = demo.columns.tolist()
print(alist.sort())#mis_automation.ds_exploration.acard_cc_data

In [0]:
# Join acard_cc_data (excluding demographic cols) with customer_information on CIF = Customer_number
from pyspark.sql.functions import col, floor, months_between

df_base = spark.table("mis_automation.ds_exploration.acard_cc_data")

# Drop existing demographic columns to avoid duplicates after join
drop_cols = ["DATE_OF_BIRTH", "AGE", "NATIONALITY", "SEX"]
existing_drop = [c for c in drop_cols if c in df_base.columns]
df_base = df_base.drop(*existing_drop)

df_base.createOrReplaceTempView("_acard_cc_base")

df_final = spark.sql("""
    SELECT
        a.*,
        c.NATIONALITY,
        c.DATE_OF_BIRTH,
        c.SEX,
        FLOOR(MONTHS_BETWEEN(a.APPROVEDATE, c.DATE_OF_BIRTH) / 12) AS AGE
    FROM _acard_cc_base a
    LEFT JOIN (
        SELECT DISTINCT Customer_number, NATIONALITY, DATE_OF_BIRTH, SEX
        FROM mis_automation.silver_fin_db.customer_information
        WHERE Customer_number IS NOT NULL
    ) c
    ON a.CIF = c.Customer_number
""")

# Save to table
df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("mis_automation.ds_exploration.acard_cc_data")
print("Saved to mis_automation.ds_exploration.acard_cc_data")



In [0]:
# spark.sql("SELECT * FROM mis_automation.ds_exploration.acard_cc_data LIMIT 5").display() #mis_automation.datascience_pbg.acard_master
spark.sql("SELECT aecb_income_age_months FROM mis_automation.datascience_pbg.acard_master LIMIT 5").display()